# Daily continuous waveform processing and ASL run

This notebook reads one day at a time from the MVO Seisan archive, removes instrument response, trims to the target day, and passes the resulting `ObsPy Stream` to `run_single_event()`.

The workflow assumes that when `run_single_event()` receives a `Stream`, the data are **already pre-processed**.

## 1. Imports

This section collects the notebook imports in one place and keeps the path setup visible near the top.

In [ ]:
import sys
from pathlib import Path

import numpy as np
from obspy import UTCDateTime, read_inventory

In [ ]:
sys.path.append("../week8")  # location of set_samba_data_root.py
from set_samba_data_root import DATA_ROOT

In [ ]:
from flovopy.processing.sam import VSAM
from flovopy.research.mvo.archive import MVOSeisanArchive

In [ ]:
from flovopy.asl.grid import Grid
from flovopy.asl.wrappers import run_single_event
from flovopy.asl.config import ASLConfig

In [ ]:
import asl_env

## 2. User settings

Keep the main knobs together so they are easy to review and change.

In [ ]:
DEBUG = False
WINDOW_SECONDS = 60          # RSAM/DSAM/VSAM window length in seconds
TAPER_SECONDS = 60 * 60      # one hour of padding on each side of the target day
PRE_FILT = [0.1, 0.2, 18, 25]  # response-removal pre-filter
SECONDS_PER_DAY = 24 * 60 * 60

## 3. Load station metadata and initialize the archive client

- `inv` is the station metadata used for response removal.
- `mvo` provides access to the Seisan continuous archive.

In [ ]:
RESPONSE_DIR = Path(DATA_ROOT) / "SEISAN_DB" / "CAL"
STATIONXML = RESPONSE_DIR / "MV.xml"
inv = read_inventory(STATIONXML)

MVO_ROOT = Path(DATA_ROOT) / "SEISAN_DB"
seisanclient = MVOSeisanArchive(MVO_ROOT)

## 4. Define source and topographic context for ASL

The source location is retained here for reference.  
The ASL plotting and mapping options are collected in `topo_kw`.

In [ ]:
source = {"lat": 16.7164, "lon": -62.1654}  # Soufrière Hills volcano reference location

gridobj = Grid.load(asl_env.GRIDFILE_DEFAULT)

topo_kw = {
    "inv": asl_env.INV,
    "add_labels": True,
    "cmap": "gray",
    "region": asl_env.REGION_DEFAULT,
    "dem_tif": asl_env.DEM_DEFAULT,
    "title": "Montserrat DEM + Stations",
    "frame": True,
    "dome_location": asl_env.DOME_LOCATION,
}



Station corrections

In [ ]:
# Montserrat station corrections estimated from regionals
import pandas as pd
station_corrections_csv = asl_env.STATION_CORRECTIONS_DIR / "station_gains_intervals.csv"
station_corrections_df = pd.read_csv(station_corrections_csv)

## 5. Build the baseline ASL configuration

This is the configuration passed to `run_single_event()` for each day.

In [ ]:
baseline_cfg = ASLConfig(
    inventory=inv,
    output_base=asl_env.OUTPUT_DIR,
    gridobj=gridobj,
    global_cache=asl_env.GLOBAL_CACHE,
    station_correction_dataframe=station_corrections_df,
    wave_kind="surface",
    speed=1.5,
    Q=23,
    peakf=2.0,
    dist_mode="3d",
    misfit_engine="r2",
    window_seconds=WINDOW_SECONDS,
    min_stations=5,
    sam_class=VSAM,
    sam_metric="mean",
    debug=DEBUG,
)

baseline_cfg.build()

## 6. Define the processing interval

The loop below processes one day at a time, with the end time treated as exclusive.

In [ ]:
start_time = UTCDateTime(2003, 7, 1)   # inclusive
end_time = UTCDateTime(2003, 7, 14)    # exclusive

num_days = (end_time - start_time) / SECONDS_PER_DAY
print(f"Processing interval: {start_time.date} to {end_time.date} ({num_days:.0f} days)")

## 7. Helper function for daily stream preparation

This helper keeps the loop cleaner and makes the preprocessing steps easier to inspect.

Processing steps:
1. Read one padded day from the archive
2. Replace NaNs
3. Detrend and merge
4. Taper
5. Remove response to velocity
6. Trim back to the exact target day

In [ ]:
def load_and_prepare_day_stream(daytime, *, seisanclient, inv, pre_filt, taper_seconds, debug=False):
    """Read and preprocess one day of continuous waveform data."""

    st = seisanclient.read_continuous_stream(
        daytime - taper_seconds,
        #daytime + SECONDS_PER_DAY + taper_seconds,
        daytime + 3600 + taper_seconds,  # one hour of data, plus the taper padding
        verbose=False,
        seismic_only=True,
        vertical_only=True,
        merge=True,
    )

    print(f"  Loaded {len(st)} traces")

    if len(st) == 0:
        return st
    
    # Check for masked numpy arrays and convert to regular arrays if needed.
    for tr in st:
        if isinstance(tr.data, np.ma.MaskedArray):
            tr.data = tr.data.filled(0)

    # Replace NaNs before response removal.
    for tr in st:
        tr.data = np.nan_to_num(tr.data, nan=0)

    st.detrend("linear")

    # Use the full stream span to define the taper fraction.
    seconds_in_stream = st[0].stats.endtime - st[0].stats.starttime
    if seconds_in_stream <= 0:
        raise ValueError("Stream duration is zero or negative.")

    st.merge(method=1, fill_value="latest")
    st.detrend("demean")
    st.taper(max_percentage=taper_seconds / seconds_in_stream)

    # Convert to velocity in m/s.
    st.remove_response(
        inventory=inv,
        pre_filt=pre_filt,
        output="VEL",
        taper=False,
        plot=DEBUG,
        #detrend=False,
    )

    # Trim off the padding and keep only the target day.
    st.trim(starttime=daytime, endtime=daytime + SECONDS_PER_DAY)

    if debug:
        print(f"  Plotting velocity seismograms for {daytime.date}")
        st.plot()

    return st

## 8. Main day-by-day processing loop

Each day is read, preprocessed to velocity, and then submitted to `run_single_event()` as an already-prepared `Stream`.

In [ ]:
daytime = start_time

while daytime < end_time:
    print("=" * 80)
    print(f"Reading day: {daytime.date}")

    
    st = load_and_prepare_day_stream(
        daytime,
        seisanclient=seisanclient,
        inv=inv,
        pre_filt=PRE_FILT,
        taper_seconds=TAPER_SECONDS,
        debug=DEBUG,
    )
    

    if len(st) == 0:
        print("  No traces found for this day. Skipping.")
        daytime += SECONDS_PER_DAY
        continue

    result = run_single_event(
        event_input=st,
        cfg=baseline_cfg,
        refine_sector=False,
        station_gains_df=None,
        switch_event_ctag=True,
        topo_kw=topo_kw,
        mseed_units="m/s",   # largely informational here because the input is already a Stream
        reduce_time=False,
        debug=DEBUG,
    )

    print("  Result keys:", list(result.keys()))
    daytime += SECONDS_PER_DAY